In [ ]:
!pip install sacrebleu

In [ ]:
import os
import math
import random
import time
from collections import Counter

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from torch.nn.utils.rnn import pad_sequence
from tqdm import tqdm
import sacrebleu

In [ ]:
class Config:
    data_dir = "/kaggle/input/datasets/bobemka/bhw-2-lol"
    train_src = os.path.join(data_dir, "train.de-en.de")
    train_tgt = os.path.join(data_dir, "train.de-en.en")
    val_src = os.path.join(data_dir, "val.de-en.de")
    val_tgt = os.path.join(data_dir, "val.de-en.en")
    test_src = os.path.join(data_dir, "test1.de-en.de")
    output_file = "test1.de-en.en"

    min_freq = 2
    max_vocab_src = 32000
    max_vocab_tgt = 32000

    d_model = 256
    nhead = 8
    num_encoder_layers = 3
    num_decoder_layers = 3
    dim_feedforward = 1024
    dropout = 0.1

    batch_size = 128
    num_epochs = 25
    lr = 3e-4
    warmup_steps = 4000
    max_len = 100
    label_smoothing = 0.1
    clip_grad = 1.0

    max_decode_len = 100
    beam_size = 1

    seed = 42
    device = "cuda" if torch.cuda.is_available() else "cpu"
    save_model = "model.pt"
    log_every = 200

cfg = Config()

random.seed(cfg.seed)
torch.manual_seed(cfg.seed)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(cfg.seed)

print(f"Device: {cfg.device}")

Device: cuda


In [ ]:
PAD_TOKEN, UNK_TOKEN, BOS_TOKEN, EOS_TOKEN = "<pad>", "<unk>", "<bos>", "<eos>"
SPECIAL_TOKENS = [PAD_TOKEN, UNK_TOKEN, BOS_TOKEN, EOS_TOKEN]
PAD_IDX, UNK_IDX, BOS_IDX, EOS_IDX = 0, 1, 2, 3


class Vocabulary:
    def __init__(self):
        self.token2idx = {}
        self.idx2token = []

    def build(self, sentences, min_freq=2, max_size=32000):
        counter = Counter()
        for sent in sentences:
            counter.update(sent)

        self.idx2token = list(SPECIAL_TOKENS)
        self.token2idx = {t: i for i, t in enumerate(SPECIAL_TOKENS)}

        for token, freq in sorted(counter.items(), key=lambda x: (-x[1], x[0])):
            if freq < min_freq:
                continue
            if len(self.idx2token) >= max_size:
                break
            if token not in self.token2idx:
                self.token2idx[token] = len(self.idx2token)
                self.idx2token.append(token)

    def encode(self, tokens):
        return [self.token2idx.get(t, UNK_IDX) for t in tokens]

    def decode(self, indices):
        tokens = []
        for idx in indices:
            if idx == EOS_IDX:
                break
            if idx in (PAD_IDX, BOS_IDX):
                continue
            tokens.append(self.idx2token[idx] if idx < len(self.idx2token) else UNK_TOKEN)
        return tokens

    def __len__(self):
        return len(self.idx2token)

In [ ]:
def read_lines(path):
    with open(path, "r", encoding="utf-8") as f:
        return [line.strip().split() for line in f]


class TranslationDataset(Dataset):
    def __init__(self, src_sents, tgt_sents, src_vocab, tgt_vocab, max_len=100):
        self.pairs = []
        for s, t in zip(src_sents, tgt_sents):
            s, t = s[:max_len], t[:max_len]
            src_ids = [BOS_IDX] + src_vocab.encode(s) + [EOS_IDX]
            tgt_ids = [BOS_IDX] + tgt_vocab.encode(t) + [EOS_IDX]
            self.pairs.append((torch.tensor(src_ids, dtype=torch.long),
                               torch.tensor(tgt_ids, dtype=torch.long)))

    def __len__(self):
        return len(self.pairs)

    def __getitem__(self, idx):
        return self.pairs[idx]


class TestDataset(Dataset):
    def __init__(self, src_sents, src_vocab, max_len=100):
        self.srcs = []
        for s in src_sents:
            s = s[:max_len]
            src_ids = [BOS_IDX] + src_vocab.encode(s) + [EOS_IDX]
            self.srcs.append(torch.tensor(src_ids, dtype=torch.long))

    def __len__(self):
        return len(self.srcs)

    def __getitem__(self, idx):
        return self.srcs[idx]


def collate_fn(batch):
    srcs, tgts = zip(*batch)
    return (pad_sequence(srcs, batch_first=True, padding_value=PAD_IDX),
            pad_sequence(tgts, batch_first=True, padding_value=PAD_IDX))


def collate_test(batch):
    return pad_sequence(batch, batch_first=True, padding_value=PAD_IDX)

In [ ]:
class PositionalEncoding(nn.Module):
    def __init__(self, d_model, max_len=5000, dropout=0.1):
        super().__init__()
        self.dropout = nn.Dropout(dropout)
        pe = torch.zeros(max_len, d_model)
        position = torch.arange(0, max_len, dtype=torch.float).unsqueeze(1)
        div_term = torch.exp(torch.arange(0, d_model, 2).float() * (-math.log(10000.0) / d_model))
        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)
        self.register_buffer("pe", pe.unsqueeze(0))

    def forward(self, x):
        x = x + self.pe[:, :x.size(1)]
        return self.dropout(x)


class TransformerTranslator(nn.Module):
    def __init__(self, src_vocab_size, tgt_vocab_size, d_model=256, nhead=8,
                 num_encoder_layers=3, num_decoder_layers=3,
                 dim_feedforward=512, dropout=0.3):
        super().__init__()
        self.d_model = d_model
        self.src_embedding = nn.Embedding(src_vocab_size, d_model, padding_idx=PAD_IDX)
        self.tgt_embedding = nn.Embedding(tgt_vocab_size, d_model, padding_idx=PAD_IDX)
        self.pos_enc = PositionalEncoding(d_model, dropout=dropout)
        self.transformer = nn.Transformer(
            d_model=d_model, nhead=nhead,
            num_encoder_layers=num_encoder_layers,
            num_decoder_layers=num_decoder_layers,
            dim_feedforward=dim_feedforward,
            dropout=dropout, batch_first=True,
        )
        self.output_proj = nn.Linear(d_model, tgt_vocab_size)
        self._init_weights()

    def _init_weights(self):
        for p in self.parameters():
            if p.dim() > 1:
                nn.init.xavier_uniform_(p)

    def encode(self, src):
        src_pad_mask = (src == PAD_IDX)
        src_emb = self.pos_enc(self.src_embedding(src) * math.sqrt(self.d_model))
        memory = self.transformer.encoder(src_emb, src_key_padding_mask=src_pad_mask)
        return memory, src_pad_mask

    def decode(self, tgt, memory, memory_key_padding_mask):
        tgt_len = tgt.size(1)
        tgt_causal = nn.Transformer.generate_square_subsequent_mask(tgt_len, device=tgt.device)
        tgt_pad_mask = (tgt == PAD_IDX)
        tgt_emb = self.pos_enc(self.tgt_embedding(tgt) * math.sqrt(self.d_model))
        output = self.transformer.decoder(
            tgt_emb, memory,
            tgt_mask=tgt_causal,
            tgt_key_padding_mask=tgt_pad_mask,
            memory_key_padding_mask=memory_key_padding_mask,
        )
        return self.output_proj(output)

    def forward(self, src, tgt):
        memory, src_pad_mask = self.encode(src)
        return self.decode(tgt, memory, src_pad_mask)

In [ ]:
def train_epoch(model, dataloader, optimizer, criterion):
    model.train()
    total_loss, total_tokens = 0, 0

    for i, (src, tgt) in enumerate(tqdm(dataloader, desc="Training")):
        src, tgt = src.to(cfg.device), tgt.to(cfg.device)
        tgt_input, tgt_output = tgt[:, :-1], tgt[:, 1:]

        logits = model(src, tgt_input)
        loss = criterion(logits.reshape(-1, logits.size(-1)), tgt_output.reshape(-1))

        optimizer.zero_grad()
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), cfg.clip_grad)
        optimizer.step()

        non_pad = (tgt_output != PAD_IDX).sum().item()
        total_loss += loss.item() * non_pad
        total_tokens += non_pad

        if (i + 1) % cfg.log_every == 0:
            print(f"Step {i+1}, Loss: {total_loss/total_tokens}")

    return total_loss / total_tokens


def evaluate(model, dataloader, criterion):
    model.eval()
    total_loss, total_tokens = 0, 0
    with torch.no_grad():
        for src, tgt in dataloader:
            src, tgt = src.to(cfg.device), tgt.to(cfg.device)
            tgt_input, tgt_output = tgt[:, :-1], tgt[:, 1:]
            logits = model(src, tgt_input)
            loss = criterion(logits.reshape(-1, logits.size(-1)), tgt_output.reshape(-1))
            non_pad = (tgt_output != PAD_IDX).sum().item()
            total_loss += loss.item() * non_pad
            total_tokens += non_pad
    return total_loss / total_tokens

In [ ]:
@torch.no_grad()
def greedy_decode(model, src, max_len, device):
    model.eval()
    src = src.to(device)
    memory, src_pad_mask = model.encode(src)
    batch_size = src.size(0)
    ys = torch.full((batch_size, 1), BOS_IDX, dtype=torch.long, device=device)
    finished = torch.zeros(batch_size, dtype=torch.bool, device=device)

    for _ in range(max_len):
        logits = model.decode(ys, memory, src_pad_mask)
        next_token = logits[:, -1, :].argmax(dim=-1)
        next_token = next_token.masked_fill(finished, PAD_IDX)
        ys = torch.cat([ys, next_token.unsqueeze(1)], dim=1)
        finished = finished | (next_token == EOS_IDX)
        if finished.all():
            break
    return ys.cpu().tolist()


def translate_file(model, dataset, tgt_vocab, batch_size=64):
    loader = DataLoader(dataset, batch_size=batch_size, shuffle=False, collate_fn=collate_test)
    translations = []
    for src_batch in tqdm(loader, desc="Translating"):
        decoded = greedy_decode(model, src_batch, cfg.max_decode_len, cfg.device)
        for seq in decoded:
            translations.append(" ".join(tgt_vocab.decode(seq)))
    return translations


def compute_bleu_on_val(model, val_src_sents, val_tgt_sents, src_vocab, tgt_vocab):
    ds = TestDataset(val_src_sents, src_vocab, max_len=cfg.max_len)
    translations = translate_file(model, ds, tgt_vocab, batch_size=128)
    references = [" ".join(s) for s in val_tgt_sents]
    bleu = sacrebleu.corpus_bleu(translations, [references], tokenize="none")
    return bleu.score, translations

In [ ]:
train_src_sents = read_lines(cfg.train_src)
train_tgt_sents = read_lines(cfg.train_tgt)
val_src_sents = read_lines(cfg.val_src)
val_tgt_sents = read_lines(cfg.val_tgt)
test_src_sents = read_lines(cfg.test_src)

print(f"Train: {len(train_src_sents)}, Val: {len(val_src_sents)}, Test: {len(test_src_sents)}")

Train: 195915, Val: 986, Test: 2998


In [ ]:
src_vocab = Vocabulary()
src_vocab.build(train_src_sents, min_freq=cfg.min_freq, max_size=cfg.max_vocab_src)

tgt_vocab = Vocabulary()
tgt_vocab.build(train_tgt_sents, min_freq=cfg.min_freq, max_size=cfg.max_vocab_tgt)

In [ ]:
train_ds = TranslationDataset(train_src_sents, train_tgt_sents, src_vocab, tgt_vocab, cfg.max_len)
val_ds = TranslationDataset(val_src_sents, val_tgt_sents, src_vocab, tgt_vocab, cfg.max_len)
test_ds = TestDataset(test_src_sents, src_vocab, cfg.max_len)

train_loader = DataLoader(train_ds, batch_size=cfg.batch_size, shuffle=True,
                          collate_fn=collate_fn, num_workers=0, drop_last=True)
val_loader = DataLoader(val_ds, batch_size=cfg.batch_size, shuffle=False,
                        collate_fn=collate_fn, num_workers=0)

model = TransformerTranslator(
    src_vocab_size=len(src_vocab),
    tgt_vocab_size=len(tgt_vocab),
    d_model=cfg.d_model,
    nhead=cfg.nhead,
    num_encoder_layers=cfg.num_encoder_layers,
    num_decoder_layers=cfg.num_decoder_layers,
    dim_feedforward=cfg.dim_feedforward,
    dropout=cfg.dropout,
).to(cfg.device)

print(f"Params: {sum(p.numel() for p in model.parameters()):,}")

optimizer = optim.Adam(model.parameters(), lr=3e-4, betas=(0.9, 0.98), eps=1e-9)
criterion = nn.CrossEntropyLoss(ignore_index=PAD_IDX, label_smoothing=cfg.label_smoothing)
scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', factor=0.5, patience=3)

Params: 30,138,624


In [ ]:
best_val_loss = float("inf")
best_bleu = 0.0

for epoch in range(1, cfg.num_epochs + 1):
    print(f"Epoch {epoch}/{cfg.num_epochs}")
    t0 = time.time()

    train_loss = train_epoch(model, train_loader, optimizer, criterion)
    val_loss = evaluate(model, val_loader, criterion)
    elapsed = time.time() - t0
    print(f"train_loss={train_loss}, val_loss={val_loss}, time={elapsed}s")

    if epoch % 3 == 0 or epoch == cfg.num_epochs or epoch >= cfg.num_epochs - 2:
        bleu_score, _ = compute_bleu_on_val(model, val_src_sents, val_tgt_sents, src_vocab, tgt_vocab)
        print(f"Val BLEU: {bleu_score}")
        if bleu_score > best_bleu:
            best_bleu = bleu_score
            torch.save(model.state_dict(), cfg.save_model)
            print("Best BLEU saved.")
    elif val_loss < best_val_loss:
        best_val_loss = val_loss
        torch.save(model.state_dict(), cfg.save_model)
        print("Best val loss saved")

Epoch 1/25


Training:  13%|█▎        | 200/1530 [00:39<04:12,  5.26it/s]

Step 200, Loss: 6.981762527850719


Training:  26%|██▌       | 401/1530 [01:18<03:29,  5.40it/s]

Step 400, Loss: 6.402414284825067


...

Step 1400, Loss: 2.7287414169112028


Training: 100%|██████████| 1530/1530 [05:00<00:00,  5.09it/s]


train_loss=2.73365037352743, val_loss=3.330849243023766, time=301.0949430465698s


Translating: 100%|██████████| 8/8 [00:10<00:00,  1.30s/it]
That's 100 lines that end in a tokenized period ('.')
It looks like you forgot to detokenize your test data, which may hurt your score.
If you insist your data is detokenized, or don't care, you can suppress this message with the `force` parameter.


Val BLEU: 31.863508316391638


In [ ]:
model.load_state_dict(torch.load(cfg.save_model, map_location=cfg.device))

final_bleu, _ = compute_bleu_on_val(model, val_src_sents, val_tgt_sents, src_vocab, tgt_vocab)
print(f"Final val BLEU: {final_bleu}")

translations = translate_file(model, test_ds, tgt_vocab)
with open(cfg.output_file, "w", encoding="utf-8") as f:
    for line in translations:
        f.write(line + "\n")

for i in range(min(5, len(translations))):
    print(f"DE: {' '.join(test_src_sents[i][:15])}...")
    print(f"EN: {translations[i][:80]}")
    print()

Translating: 100%|██████████| 8/8 [00:10<00:00,  1.34s/it]
That's 100 lines that end in a tokenized period ('.')
It looks like you forgot to detokenize your test data, which may hurt your score.
If you insist your data is detokenized, or don't care, you can suppress this message with the `force` parameter.


Final val BLEU: 31.90490040523673


Translating: 100%|██████████| 47/47 [00:27<00:00,  1.72it/s]

DE: und das funktionierte . applaus für scott ....
EN: and that worked . it worked for scott .

DE: und im verlauf der nächsten acht oder neun tage , ergaben sie sich dem geschehenen...
EN: and in the course of the next eight or nine days , they evolved to be able to do

DE: vielen dank ....
EN: thank you very much .

DE: nun haben wir » drei-verstöße « -gesetze , die menschen für immer hinter gitter bringen...
EN: now we have " <unk> " <unk> , which brings people to the media and the people be

DE: wir besitzen einen kognitiven fehler , so dass wir einen perfekten krankheitsverlauf für einen patienten...
EN: we have a cognitive error so that we can take a perfect <unk> for a patient with

